<a href="https://colab.research.google.com/github/KiaNoForte/Cryptography/blob/main/CryptoLab12_Shannon_to_Modern_Security.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CryptoLab 12 — From Shannon to Modern Security

**CYBR 3570 Cryptography**

This lab moves from the one-time pad and Shannon-style perfect secrecy to modern adversarial security games. The goal is **not** to write formal proofs. Instead, you will use small experiments to see what different security claims actually mean.

> **Important:** Several constructions in this notebook use tiny parameters or simplified mathematics so the behavior is visible. They are teaching models, not production cryptography.

## Learning Objectives

By the end of this notebook, you should be able to:

- explain the difference between perfect secrecy and computational security;
- demonstrate why a correctly used one-time pad hides the plaintext;
- demonstrate why OTP key reuse is catastrophic;
- model an indistinguishability-style security game;
- explain why deterministic encryption fails under repeated/chosen plaintexts;
- compare deterministic and randomized encryption behavior;
- model the idea behind ElGamal randomized public-key encryption;
- demonstrate ciphertext malleability and explain why authenticated encryption matters;
- write a simple security claim in terms of a goal, adversary, and assumptions.

In [1]:
import os
import random
import secrets
from collections import Counter
from typing import Callable

from cryptography.hazmat.primitives.ciphers.aead import AESGCM

print("Notebook ready.")

Notebook ready.


# Part 1 — Perfect Secrecy in a Tiny World

For a one-time pad, encryption is:

```text
C = M XOR K
```

If the key is uniformly random, the same length as the message, and used exactly once, every ciphertext is compatible with every plaintext through exactly one key.

In [2]:
def xor_int(a: int, b: int) -> int:
    return a ^ b

messages = list(range(4))   # 2-bit messages: 00, 01, 10, 11
keys = list(range(4))       # 2-bit keys

# Show all possible encryptions.
for m in messages:
    row = [xor_int(m, k) for k in keys]
    print(f"M={m:02b} ->", [f"{c:02b}" for c in row])

M=00 -> ['00', '01', '10', '11']
M=01 -> ['01', '00', '11', '10']
M=10 -> ['10', '11', '00', '01']
M=11 -> ['11', '10', '01', '00']


### Exercise 1 — What does the ciphertext tell you?

Choose any ciphertext value from `00` through `11`. For each possible plaintext, determine the unique key that would produce that ciphertext.

Then answer:

1. After observing the ciphertext, can the attacker rule out any plaintext?
2. Why does this illustrate perfect secrecy?

In [5]:
observed_c = 0b10

# TODO: Fill this dictionary with plaintext -> required key.
possible_explanations = {}

for m in messages:
    # required_key = ?
    required_key = xor_int(m, observed_c)
    possible_explanations[f"m={m:02b}"] = f"{required_key:02b}"
    pass

possible_explanations

{'m=00': '10', 'm=01': '11', 'm=10': '00', 'm=11': '01'}

# Part 2 — The One-Time Pad Fails When the Key Is Reused

If the same pad encrypts two messages:

```text
C1 = M1 XOR K
C2 = M2 XOR K
```

then:

```text
C1 XOR C2 = M1 XOR M2
```

The key cancels out.

In [6]:
def xor_bytes(a: bytes, b: bytes) -> bytes:
    return bytes(x ^ y for x, y in zip(a, b))

m1 = b"ATTACK AT DAWN!!"
m2 = b"RETREAT AT NOON!"
key = secrets.token_bytes(len(m1))

c1 = xor_bytes(m1, key)
c2 = xor_bytes(m2, key)

print("C1 XOR C2:", xor_bytes(c1, c2).hex())
print("M1 XOR M2:", xor_bytes(m1, m2).hex())
print("Equal?", xor_bytes(c1, c2) == xor_bytes(m1, m2))

C1 XOR C2: 13110013060a74611574640f18016f00
M1 XOR M2: 13110013060a74611574640f18016f00
Equal? True


### Exercise 2 — Known plaintext recovery

Assume an attacker learns `m1` and has captured both `c1` and `c2`. Recover `m2` without directly using the original `key` variable.

Explain why this attack does **not** contradict the proof of one-time-pad security.

In [7]:
# TODO: Recover the reused pad from m1 and c1, then recover m2.
recovered_key = xor_bytes(c1, c2)
recovered_m2 = xor_bytes(m1, recovered_key)

print("Recovered message:", recovered_m2)

Recovered message: b'RETREAT AT NOON!'


# Part 3 — From Perfect Secrecy to Computational Security

Perfect secrecy says the attacker learns nothing even with unlimited computation. Modern cryptography usually makes a weaker but much more practical claim: attacks may exist in principle, but no **efficient** attacker should succeed with meaningful probability.

The work factor grows exponentially with key size for an ideal brute-force search.

In [8]:
for bits in [40, 56, 80, 128, 256]:
    keyspace = 2 ** bits
    print(f"{bits:3d}-bit key -> {keyspace:.3e} possible keys")

 40-bit key -> 1.100e+12 possible keys
 56-bit key -> 7.206e+16 possible keys
 80-bit key -> 1.209e+24 possible keys
128-bit key -> 3.403e+38 possible keys
256-bit key -> 1.158e+77 possible keys


### Exercise 3 — Security is a claim with a budget

Suppose an attacker can test `10^12` keys per second. Estimate the time to search **half** of a 56-bit, 80-bit, and 128-bit key space. Express your answers in years.

What does this tell you about the phrase “computationally infeasible”?

In [9]:
tests_per_second = 10**12
seconds_per_year = 365.25 * 24 * 60 * 60

# TODO: Calculate expected brute-force years for each key size.
estimates = {}
for bits in [56, 80, 128]:
    half_keyspace = (2 ** bits) / 2
    seconds = half_keyspace / tests_per_second
    years = seconds / seconds_per_year
    estimates[f"{bits}-bit"] = f"{years:.3e} years"
    pass

estimates

{'56-bit': '1.142e-03 years',
 '80-bit': '1.915e+04 years',
 '128-bit': '5.391e+18 years'}

# Part 4 — An Indistinguishability Game

A security game makes the attacker's task explicit. In a left-or-right game:

1. The attacker chooses equal-length messages `M0` and `M1`.
2. The challenger secretly chooses a bit `b`.
3. The challenger encrypts `Mb`.
4. The attacker guesses `b`.

Random guessing succeeds about 50% of the time.

In [11]:
def left_or_right_game(encrypt: Callable[[bytes], bytes], adversary, rounds=1000):
    wins = 0
    for _ in range(rounds):
        m0, m1 = adversary.choose_messages()
        if len(m0) != len(m1):
            raise ValueError("Messages must have the same length")
        b = secrets.randbelow(2)
        c = encrypt(m0 if b == 0 else m1)
        guess = adversary.guess(c)
        wins += int(guess == b)
    return wins / rounds

class RandomGuessAdversary:
    def choose_messages(self):
        return b"PAY=0000", b"PAY=9999"
    def guess(self, ciphertext):
        return secrets.randbelow(2)

### Exercise 4 — Establish a baseline

Create an encryption function that returns random bytes of the same length as its input. Run the game with `RandomGuessAdversary`.

You should get a success rate near 0.5. Why is 0.5 the baseline rather than 0?

In [14]:
def random_world_encrypt(message: bytes) -> bytes:
    # TODO: Return uniformly random bytes with the same length as message.
    return secrets.token_bytes(len(message))

# TODO: Run left_or_right_game(...) and record the result.
baseline_win_rate = left_or_right_game(random_world_encrypt, RandomGuessAdversary())
print("Baseline success rate:", baseline_win_rate)

Baseline success rate: 0.508


# Part 5 — Deterministic Encryption Leaks Equality

A deterministic encryption function always produces the same ciphertext for the same key and plaintext. Even if the ciphertext is unreadable, repeated plaintexts produce repeated ciphertexts.

In [15]:
# A toy deterministic "encryption" based on a fixed secret permutation of byte values.
perm = list(range(256))
random.Random(3570).shuffle(perm)

def deterministic_toy_encrypt(message: bytes) -> bytes:
    return bytes(perm[b] for b in message)

for message in [b"YES", b"NO!", b"YES"]:
    print(message, deterministic_toy_encrypt(message).hex())

b'YES' 74d529
b'NO!' 45de0f
b'YES' 74d529


### Exercise 5 — Build a chosen-plaintext distinguisher

An attacker is allowed to ask for an encryption of `M0` before receiving the challenge ciphertext. Use that capability to distinguish whether the challenge encrypted `M0` or `M1` under deterministic encryption.

Write down the attack in plain English, then implement the comparison below.

In [16]:
m0 = b"PAY=0000"
m1 = b"PAY=9999"
reference = deterministic_toy_encrypt(m0)

# Simulate a challenge.
b = secrets.randbelow(2)
challenge = deterministic_toy_encrypt(m0 if b == 0 else m1)

# TODO: Guess 0 if challenge equals the known encryption of m0; otherwise guess 1.
guess = 0 if challenge == reference else 1
print("Secret bit:", b, "Guess:", guess)

Secret bit: 1 Guess: 1


# Part 6 — Randomization Breaks Equality Patterns

Modern encryption schemes normally use fresh randomness or a unique nonce so that encrypting the same plaintext twice does not produce the same ciphertext.

In [17]:
aes_key = AESGCM.generate_key(bit_length=128)
aesgcm = AESGCM(aes_key)

def randomized_encrypt(message: bytes) -> bytes:
    nonce = os.urandom(12)
    return nonce + aesgcm.encrypt(nonce, message, None)

for _ in range(3):
    print(randomized_encrypt(b"PAY=0000").hex())

399ab8a81088af4e1f3d7e4f42ca58368a2d6503203c81d11a2491e78e3fc5633a949a53
a7710510d57c149c3547de3636736ed50b9d0f016eb15ad70c93eb8a89054817395ce19e
f735620dd34c0bd3584e80933f7f357c6878a623b7ce06303f59fb873a23bcc32ab76080


### Exercise 6 — Compare observable patterns

Encrypt the same message 100 times with `deterministic_toy_encrypt` and 100 times with `randomized_encrypt`. Count the number of distinct ciphertexts in each set.

Explain what equality information is exposed in each case.

In [18]:
# TODO: Build both ciphertext sets and count distinct values.
deterministic_outputs = [deterministic_toy_encrypt(b"PAY=0000") for _ in range(100)]
randomized_outputs = [randomized_encrypt(b"PAY=0000") for _ in range(100)]

print("Distinct deterministic:", len(set(deterministic_outputs)))
print("Distinct randomized:", len(set(randomized_outputs)))

Distinct deterministic: 1
Distinct randomized: 100


# Part 7 — Toy ElGamal: Randomized Public-Key Encryption

ElGamal is closely related to Diffie–Hellman. We will use tiny numbers to make the algebra visible.

For public parameters `(p, g)`:

```text
private key: x
public key:  y = g^x mod p

Encrypt message m using fresh random k:
c1 = g^k mod p
c2 = m * y^k mod p

Decrypt:
m = c2 * (c1^x)^(-1) mod p
```

Real implementations use carefully selected groups and message encodings. This tiny version is for learning only.

In [19]:
p = 467
g = 2
x = 127
y = pow(g, x, p)

def elgamal_encrypt(m: int):
    if not 1 <= m < p:
        raise ValueError("Message must be a nonzero group element")
    k = secrets.randbelow(p - 2) + 1
    c1 = pow(g, k, p)
    shared = pow(y, k, p)
    c2 = (m * shared) % p
    return c1, c2

def elgamal_decrypt(ciphertext):
    c1, c2 = ciphertext
    shared = pow(c1, x, p)
    shared_inv = pow(shared, -1, p)
    return (c2 * shared_inv) % p

message = 123
for _ in range(3):
    ct = elgamal_encrypt(message)
    print(ct, "->", elgamal_decrypt(ct))

(430, 173) -> 123
(78, 148) -> 123
(145, 220) -> 123


### Exercise 7 — Why does the same plaintext look different?

Encrypt the same integer 20 times. Count the number of distinct `(c1, c2)` pairs.

Then explain which value provides the per-encryption randomness and how this resembles ephemeral Diffie–Hellman.

In [20]:
# TODO: Collect 20 ElGamal ciphertexts for the same message.
ciphertexts = [elgamal_encrypt(123) for _ in range(20)]
print("Distinct ciphertexts:", len(set(ciphertexts)))

Distinct ciphertexts: 18


# Part 8 — Malleability: Confidentiality Is Not Integrity

A stream-style encryption that XORs plaintext with a keystream can hide content while still allowing predictable modification. If an attacker flips a bit in the ciphertext, the corresponding plaintext bit flips after decryption.

In [21]:
plain = bytearray(b"admin=0;amount=100")
stream_key = secrets.token_bytes(len(plain))
cipher = bytearray(xor_bytes(bytes(plain), stream_key))

# Flip the ASCII bit that changes '0' (0x30) to '1' (0x31).
index = plain.index(ord("0"))
cipher[index] ^= 0x01

tampered_plain = xor_bytes(bytes(cipher), stream_key)
print(tampered_plain)

b'admin=1;amount=100'


### Exercise 8 — Compare with authenticated encryption

Use AES-GCM to encrypt `b"admin=0;amount=100"`. Flip one bit in the ciphertext and attempt to decrypt it.

What happens? Why is this failure behavior a security feature?

In [23]:
message = b"admin=0;amount=100"
nonce = os.urandom(12)
protected = bytearray(aesgcm.encrypt(nonce, message, None))

# TODO: Flip one bit in protected, then attempt aesgcm.decrypt(...).
protected[0] ^= 0x01

try:
    aesgcm.decrypt(nonce, bytes(protected), None)
except Exception:
  print("Tampering detected")
# Catch the exception and print a simple message such as "Tampering detected".


Tampering detected


# Part 9 — Write the Security Claim

A useful cryptographic claim should identify at least three things:

1. **Goal** — What property are we trying to preserve?
2. **Adversary** — What can the attacker observe, choose, modify, or compromise?
3. **Assumptions** — What must remain true for the claim to hold?

### Exercise 9 — Secure notes scenario

Your final-project team is building a secure notes application. Notes are encrypted with AES-GCM using a key derived from the user's master password.

Write a concise threat-model statement that addresses:

- the confidentiality goal;
- the integrity goal;
- what the attacker can read or modify;
- what the attacker does **not** initially know;
- at least two things the system does **not** protect against.

* Confidentiality Goal: Prevent unauthorized parties from accessing the contents of stored notes. This is achieved through computational guarantees provided by AES-GCM encryption.
* Integrity Goal: Detect tampering or unauthorized modification of ciphertext or authentication tags, and ensure that corrupted notes are rejected during decryption.
* Attacker Capabilities: The attacker is assumed to have the ability to read all stored ciphertext and authentication tags on disk, as well as intercept or modify data during transit or storage.
* Attacker Limitations: The attacker does not initially know the user's master password. Unprotected Boundaries (Limitations): The system does not defend against master-password compromise, such as weak passwords vulnerable to offline brute-force or dictionary attacks. It also does not protect against malware or keyloggers operating locally on the user's device while the app is unlocked.



# Final Reflection

Answer in 1–3 sentences each.

1. Why is perfect secrecy stronger than the security we usually expect from AES?
2. Why is key reuse fatal for a one-time pad but normal for a modern multi-use encryption scheme?
3. What does an indistinguishability game force us to specify about an attacker?
4. Why is randomized or nonce-based encryption usually preferable to deterministic encryption?
5. Why does authenticated encryption represent a stronger system building block than confidentiality-only encryption?

* Perfect secrecy, also known as Shannon security, provides a mathematical guarantee that even an attacker with unlimited computational resources and time learns nothing about the plaintext from the ciphertext. In contrast, AES provides computational security, meaning that while breaking it is theoretically possible with sufficient time or resources, it remains practically infeasible.
* Key reuse is fatal for a one-time pad because it relies on a fresh, random keystream to mask the plaintext. Reusing the key cancels out its protective effect, fully exposing the plaintexts. In contrast, modern multi-use encryption schemes such as AES-GCM use fresh nonces, initialization vectors, or randomized modes for each encryption operation, ensuring the same key produces unique ciphertexts each time.
* An indistinguishability game requires explicit specification of the attacker's operational capabilities, such as what they can observe, choose, or modify (for example, chosen-plaintext or chosen-ciphertext access), as well as their quantitative advantage over random guessing.
* Randomized or nonce-based encryption is generally preferable to deterministic encryption because it ensures that encrypting the same plaintext multiple times produces distinct ciphertexts. This approach prevents attackers from detecting repetition, frequency patterns, or equality leaks in encrypted traffic.
* Authenticated encryption represents a stronger system building block than confidentiality-only encryption because confidentiality-only encryption conceals message content but remains susceptible to ciphertext malleability, such as unauthorized tampering or bit-flipping attacks. Authenticated encryption combines confidentiality and integrity, automatically rejecting manipulated ciphertexts and preventing attackers from altering data in transit without detection.

---
## References

- Mike Rosulek, *The Joy of Cryptography*, Chapter 1: https://joyofcryptography.com/otp/
- Mike Rosulek, *The Joy of Cryptography*, Chapter 4: https://joyofcryptography.com/comp/
- Mike Rosulek, *The Joy of Cryptography*, Chapter 8: https://joyofcryptography.com/cpa/
- Mike Rosulek, *The Joy of Cryptography*, Chapter 9, *Chosen-Ciphertext Attacks Against Encryption*.

**Do not use the toy OTP, substitution, or ElGamal constructions in this notebook to protect real data.**